# CosineAnnealingWarmRestartsDecay Visualization

This notebook visualizes our custom learning rate scheduler that combines:
- **Cosine Annealing**: Smooth LR transitions within each cycle
- **Warm Restarts**: Multiple training cycles for better exploration
- **LR Decay**: Reduces max LR after each restart for stability in later phases
- **Smooth Ending**: Automatically adjusts cycle length so training ends smoothly at eta_min

This scheduler is designed to align with progressive augmentation, where:
- Early phases (high LR) -> model learns basic features with no/light augmentation
- Later phases (lower LR) -> model fine-tunes with stronger augmentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set style for better visualization
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

In [ ]:
class CosineAnnealingWarmRestartsDecay:
    """
    Cosine annealing scheduler with warm restarts and optional LR decay.

    When total_epochs is provided, the scheduler computes an effective cycle length
    so that all cycles are equal and training ends smoothly at eta_min.

    Args:
        base_lr: Base learning rate
        T_0: Approximate number of epochs per cycle (used to determine num_cycles)
        T_mult: Factor to increase cycle length after each restart (default: 1)
        eta_min: Minimum learning rate (default: 1e-6)
        decay: Factor to multiply max LR after each restart (default: 1.0, no decay)
        total_epochs: Total training epochs. Required for smooth ending.
    """

    def __init__(self, base_lr, T_0, T_mult=1, eta_min=1e-6, decay=1.0, total_epochs=None):
        self.base_lr = base_lr
        self.T_0 = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min
        self.decay = decay
        self.total_epochs = total_epochs
        
        # Compute effective cycle length for smooth ending
        self._effective_T = None
        self._num_cycles = None
        if total_epochs is not None and T_mult == 1:
            self._num_cycles = int(np.ceil(total_epochs / T_0))
            self._effective_T = total_epochs / self._num_cycles

    def get_lr(self, epoch):
        """Calculate learning rate for given epoch"""
        if self._effective_T is not None:
            # Use effective cycle length for smooth curves
            cycle = int(epoch / self._effective_T)
            T_cur = epoch - cycle * self._effective_T
            T_i = self._effective_T
        elif epoch >= self.T_0:
            if self.T_mult == 1:
                cycle = epoch // self.T_0
                T_cur = epoch % self.T_0
                T_i = self.T_0
            else:
                n = int(np.log((epoch / self.T_0 * (self.T_mult - 1) + 1)) / np.log(self.T_mult))
                cycle = n
                T_cur = epoch - self.T_0 * (self.T_mult ** n - 1) // (self.T_mult - 1)
                T_i = self.T_0 * self.T_mult ** n
        else:
            T_cur = epoch
            cycle = 0
            T_i = self.T_0

        decay_factor = self.decay ** cycle
        lr = self.eta_min + (self.base_lr * decay_factor - self.eta_min) * 0.5 * (1 + np.cos(np.pi * T_cur / T_i))
        return lr, cycle

## 1. The Problem: LR Spike at End of Training

When `epochs` doesn't divide evenly by `T_0`, the scheduler starts a new cycle at the end.

In [ ]:
# Parameters
epochs = 50
base_lr = 1e-4
T_0 = epochs // 4  # = 12
eta_min = 1e-6

print(f"epochs={epochs}, T_0={T_0}")
print(f"Problem: 50 / 12 = 4.16... cycles")
print(f"Without fix: cycles at 0-11, 12-23, 24-35, 36-47, then NEW CYCLE at 48!")

# Old behavior (without total_epochs)
scheduler_old = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=0.8, eta_min=eta_min, total_epochs=None)
lrs_old = [scheduler_old.get_lr(e)[0] for e in range(epochs)]

# New behavior (with total_epochs - smooth cycles)
scheduler_new = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=0.8, eta_min=eta_min, total_epochs=epochs)
lrs_new = [scheduler_new.get_lr(e)[0] for e in range(epochs)]

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Old behavior
axes[0].plot(range(epochs), lrs_old, 'r-', linewidth=2)
axes[0].axvline(x=48, color='red', linestyle='--', alpha=0.7)
axes[0].annotate('Unwanted spike!', xy=(48, lrs_old[48]), xytext=(38, base_lr*0.7),
                 arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Learning Rate', fontsize=12)
axes[0].set_title('WITHOUT total_epochs (Problem)', fontsize=13, color='red')
axes[0].set_xlim(0, epochs)
axes[0].set_ylim(0, base_lr * 1.1)

# New behavior
axes[1].plot(range(epochs), lrs_new, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('WITH total_epochs (Smooth)', fontsize=13, color='green')
axes[1].set_xlim(0, epochs)
axes[1].set_ylim(0, base_lr * 1.1)

plt.tight_layout()
plt.show()

print(f"\nOld: LR at epoch 47 = {lrs_old[47]:.2e}, epoch 48 = {lrs_old[48]:.2e} (SPIKE!)")
print(f"New: LR at epoch 48 = {lrs_new[48]:.2e}, epoch 49 = {lrs_new[49]:.2e} (ends smooth at eta_min)")

## 2. How It Works: Effective Cycle Length

Instead of using `T_0` directly, we compute an effective cycle length that divides evenly into `total_epochs`.

In [ ]:
print(f"T_0 = {T_0}, total_epochs = {epochs}")
print(f"\nCalculation:")
print(f"  num_cycles = ceil({epochs} / {T_0}) = ceil({epochs/T_0:.2f}) = {int(np.ceil(epochs/T_0))}")
print(f"  effective_T = {epochs} / {int(np.ceil(epochs/T_0))} = {epochs / int(np.ceil(epochs/T_0)):.2f} epochs per cycle")
print(f"\nResult: {int(np.ceil(epochs/T_0))} equal cycles of {epochs / int(np.ceil(epochs/T_0)):.2f} epochs each")
print(f"All cycles are smooth cosines ending exactly at eta_min!")

# Verify
scheduler = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=0.8, eta_min=eta_min, total_epochs=epochs)
print(f"\nScheduler computed:")
print(f"  _num_cycles = {scheduler._num_cycles}")
print(f"  _effective_T = {scheduler._effective_T}")

## 3. Decay Comparison: decay=1.0 vs decay=0.8

In [ ]:
scheduler_no_decay = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=1.0, eta_min=eta_min, total_epochs=epochs)
scheduler_decay = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=0.8, eta_min=eta_min, total_epochs=epochs)

lrs_no_decay = [scheduler_no_decay.get_lr(e)[0] for e in range(epochs)]
lrs_decay = [scheduler_decay.get_lr(e)[0] for e in range(epochs)]

effective_T = scheduler_decay._effective_T
num_cycles = scheduler_decay._num_cycles

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(range(epochs), lrs_no_decay, 'b-', linewidth=2, label='decay=1.0 (no decay)', alpha=0.7)
ax.plot(range(epochs), lrs_decay, 'r-', linewidth=2, label='decay=0.8')

# Add cycle boundaries
for i in range(1, num_cycles):
    boundary = i * effective_T
    ax.axvline(x=boundary, color='gray', linestyle='--', alpha=0.5)
    ax.text(boundary + 0.3, base_lr * 1.05, f'Cycle {i}', fontsize=10, color='gray')

# Add augmentation level annotations
aug_colors = ['#e8f5e9', '#c8e6c9', '#a5d6a7', '#81c784', '#66bb6a']
aug_labels = ['Level 0', 'Level 1', 'Level 2', 'Level 3', 'Level 4']
for i in range(num_cycles):
    start = i * effective_T
    end = (i + 1) * effective_T
    ax.axvspan(start, end, alpha=0.3, color=aug_colors[i % len(aug_colors)])
    if i < 4:  # Only show first 4 labels
        ax.text((start + end) / 2, base_lr * 0.5, aug_labels[i], ha='center', fontsize=9, color='darkgreen')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title(f'CosineAnnealingWarmRestartsDecay (epochs={epochs}, T_0={T_0}, effective_T={effective_T:.1f})', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.set_xlim(0, epochs)
ax.set_ylim(0, base_lr * 1.15)

plt.tight_layout()
plt.show()

## 4. Different Decay Values

In [ ]:
decay_values = [1.0, 0.9, 0.8, 0.7, 0.5]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(decay_values)))

fig, ax = plt.subplots(figsize=(14, 6))

for decay, color in zip(decay_values, colors):
    scheduler = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=decay, eta_min=eta_min, total_epochs=epochs)
    lrs = [scheduler.get_lr(e)[0] for e in range(epochs)]
    ax.plot(range(epochs), lrs, color=color, linewidth=2, label=f'decay={decay}')

# Add cycle boundaries
for i in range(1, num_cycles):
    ax.axvline(x=i * effective_T, color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Effect of Different Decay Values (All End Smoothly)', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.set_xlim(0, epochs)
ax.set_ylim(0, base_lr * 1.1)

plt.tight_layout()
plt.show()

## 5. Works with Any Epoch Count

In [ ]:
# Test with different epoch counts - all end smoothly!
test_epochs = [50, 55, 60, 100, 123]

fig, axes = plt.subplots(1, len(test_epochs), figsize=(20, 4))

for ax, ep in zip(axes, test_epochs):
    t0 = ep // 4
    scheduler = CosineAnnealingWarmRestartsDecay(base_lr, t0, decay=0.8, eta_min=eta_min, total_epochs=ep)
    lrs = [scheduler.get_lr(e)[0] for e in range(ep)]
    
    ax.plot(range(ep), lrs, 'b-', linewidth=1.5)
    ax.set_title(f'epochs={ep}\nT_0={t0}, eff_T={scheduler._effective_T:.1f}', fontsize=10)
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_xlim(0, ep)
    ax.set_ylim(0, base_lr * 1.1)
    
    # Verify last LR is near eta_min
    last_lr = lrs[-1]
    ax.axhline(y=eta_min, color='r', linestyle='--', alpha=0.3)
    color = 'green' if abs(last_lr - eta_min) < 1e-7 else 'red'
    ax.text(ep * 0.05, eta_min * 3, f'Final: {last_lr:.1e}', fontsize=8, color=color)

plt.suptitle('Scheduler ends smoothly at eta_min for ANY epoch count', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Max LR Per Cycle

In [ ]:
decay = 0.8
num_cycles_show = 8

max_lrs = [base_lr * (decay ** i) for i in range(num_cycles_show)]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(range(num_cycles_show), max_lrs, color=plt.cm.Reds(np.linspace(0.9, 0.3, num_cycles_show)), edgecolor='darkred')

for i, (bar, lr) in enumerate(zip(bars, max_lrs)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + base_lr*0.02, 
            f'{lr:.2e}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('Max Learning Rate', fontsize=12)
ax.set_title(f'Peak LR Per Cycle (base_lr={base_lr:.0e}, decay={decay})', fontsize=14)
ax.set_xticks(range(num_cycles_show))
ax.set_xticklabels([f'Cycle {i}' for i in range(num_cycles_show)])

plt.tight_layout()
plt.show()

print("Max LR reduction per cycle:")
for i, lr in enumerate(max_lrs):
    pct = (lr / base_lr) * 100
    print(f"  Cycle {i}: {lr:.2e} ({pct:.1f}% of base LR)")

## Summary

**CosineAnnealingWarmRestartsDecay** features:

1. **Cosine Annealing**: Smooth LR reduction within each cycle

2. **Warm Restarts**: Multiple exploration phases

3. **Decay Factor**: Reduces max LR after each restart

4. **Smooth Ending**: When `total_epochs` is provided:
   - Computes `num_cycles = ceil(total_epochs / T_0)`
   - Computes `effective_T = total_epochs / num_cycles`
   - All cycles have equal length and end smoothly at `eta_min`
   - No unwanted LR spikes regardless of epoch count!

This ensures training always ends at the minimum LR for stable convergence.